# ⚡ ApexRAG — Agentic Document Navigation

This notebook demonstrates the full ApexRAG pipeline:
1. **Ingest** a document (raw text or file)
2. **Explore** the structural tree
3. **Query** with agentic navigation
4. **Inspect** the navigation trace

> **Prerequisites:** Install `pip install apex-rag` and have [Ollama](https://ollama.com) running.

In [ ]:
# For displaying in notebooks
from IPython.display import Markdown, display

from apex_rag import ApexIndex

## 1. Create a Sample Document

Let's create a structured financial report to test with.

In [ ]:
SAMPLE_DOCUMENT = """\
# Annual Financial Report 2024

This report summarises the financial performance of Apex Corp for fiscal year 2024.

## Executive Summary

Apex Corp achieved record revenue growth of 34% year-over-year, driven primarily
by expansion in the Asia-Pacific region and the successful launch of three new
product lines in Q2 2024.

## Revenue Analysis

### Q1 2024 Revenue
Q1 revenue reached $142M, up from $98M in Q1 2023.

### Q2 2024 Revenue
Q2 marked the highest quarterly revenue in company history at $187M.

### Q3 2024 Revenue
Q3 revenue was $165M, remained 28% above Q3 2023 figures.

### Q4 2024 Revenue
Q4 closed at $198M, completing a record year with total annual revenue of $692M.

## Operating Expenses

R&D expenditure increased to $89M in 2024 (13% of revenue).
Sales and marketing spend was $134M, a 22% increase from 2023.
"""

print(f"Sample document created ({len(SAMPLE_DOCUMENT)} chars)")

## 2. Initialize ApexIndex

In [ ]:
async def setup():
    index = await ApexIndex.create(
        db_url="sqlite+aiosqlite:///apex_notebook.db",
        model="llama3.1",
        trace_enabled=True,
    )
    return index

index = await setup()
print("✅ ApexIndex initialized")

## 3. Ingest the Document

The document is parsed into a structural tree. Each heading becomes a node, and summaries are generated by the LLM.

In [ ]:
doc_id = await index.ingest_text(
    SAMPLE_DOCUMENT,
    doc_id="annual-report-2024",
    synthesize_summaries=True,
)
print(f"✅ Document ingested: doc_id={doc_id}")

## 4. Explore the Document Tree

In [ ]:
tree = await index.get_tree(doc_id)
print(f"Total nodes: {len(tree)}\n")

for node in tree:
    indent = "  " * node["depth"]
    icon = "📄" if node["is_leaf"] else "📂"
    print(f"{indent}{icon} {node['title']}  (path={node['path']})")
    if node["summary"]:
        print(f"{indent}   Summary: {node['summary'][:80]}...")

## 5. Query with Agentic Navigation

The LLM agent navigates the tree to find the exact answer.

In [ ]:
async def query(question: str):
    print(f"\n🔍 Query: {question}")
    result = await index.query(question, doc_id)

    if result:
        display(Markdown(f"**✅ Found** in **{result.title}** (path={result.path})"))
        display(Markdown(f"> {result.content[:300]}"))
        print(f"\nConfidence: {result.confidence:.2%}")
        print(f"Verified: {result.verified}")
        print("\nNavigation trace:")
        for nid, title in result.trace:
            print(f"  → {title} (id={nid})")
    else:
        print("❌ No answer found.")

await query("What was the Q3 revenue figure?")
await query("How much did R&D spending increase?")

## 6. View the Page Index

In [ ]:
entries = await index.get_page_index(doc_id)
print(f"Index entries: {len(entries)}\n")
for entry in entries[:10]:
    print(f"  {entry['term']} → path={entry['path']}")

## 7. Cleanup

In [ ]:
await index.delete(doc_id)
await index.close()
print("✅ Cleanup complete")